# Section 6: Using the real DART system

*(Replaces DART_LAB slide deck Section 6.)*

Everything in notebooks 1-5 — EAKF/EnKF/RHF, regression, localization,
fixed and adaptive inflation, QCEFF/bounded filters — exists in production
form in [DART](https://github.com/NCAR/DART), NCAR's Fortran data
assimilation framework. This notebook maps the tutorial concepts onto real
DART runs with the Lorenz 96 model.

> The shell cells below require a DART checkout and a Fortran compiler.
> Set `DART` to your DART directory to run them; otherwise read along —
> expected output is described in the text.

In [ ]:
import os
DART = os.environ.get("DART", os.path.expanduser("~/DART"))
HAVE_DART = os.path.isdir(os.path.join(DART, "models", "lorenz_96", "work"))
print(f"DART = {DART}  (found: {HAVE_DART})")

## Building DART for Lorenz 96

```bash
cd $DART/models/lorenz_96/work
./quickbuild.sh nompi
```

This builds (among others):

* `perfect_model_obs` — advances a *truth* run and harvests synthetic
  observations from it (an **OSSE**: Observing System Simulation
  Experiment, exactly what the Lorenz 96 tools did),
* `filter` — the ensemble assimilation program.

In [ ]:
if HAVE_DART:
    work = os.path.join(DART, "models", "lorenz_96", "work")
    print(os.popen(f"ls {work} | head -20").read())

## Running an OSSE

```bash
cd $DART/models/lorenz_96/work
./perfect_model_obs   # creates obs_seq.out, true_state.nc
./filter              # creates preassim.nc, analysis.nc, obs_seq.final
```

The diagnostic files are NetCDF:

* `preassim.nc` — prior ensemble at every assimilation time,
* `analysis.nc` — posterior ensemble,
* `true_state.nc` — the truth (only available in OSSEs!).

The cell below reproduces the tutorial's error/spread plot from these
files with plain Python — the equivalent of DART's MATLAB
`plot_total_err`.

In [ ]:
if HAVE_DART:
    import numpy as np
    import matplotlib.pyplot as plt
    from netCDF4 import Dataset  # noqa: F401  (xarray also works)

    work = os.path.join(DART, "models", "lorenz_96", "work")
    with Dataset(os.path.join(work, "preassim.nc")) as nc, \
         Dataset(os.path.join(work, "true_state.nc")) as tnc:
        # state_variable dims: (time, member, location)
        ens = nc["state"][:]
        truth = tnc["state"][:, 0, :]
        mean = ens.mean(axis=1)
        err = np.sqrt(((mean - truth) ** 2).mean(axis=-1))
        spread = np.sqrt((ens.std(axis=1, ddof=1) ** 2).mean(axis=-1))
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(err, label="prior RMS error")
    ax.plot(spread, label="prior spread")
    ax.set_xlabel("assimilation cycle"); ax.legend()
else:
    print("Set DART to a built DART checkout to run this cell.")

## Controlling the filter: `input.nml`

Every knob from the tutorial maps to a namelist entry:

| Tutorial control | namelist | entry |
|---|---|---|
| Localization half-width | `&assim_tools_nml` | `cutoff` |
| Ensemble size | `&filter_nml` | `ens_size` |
| Inflation flavor | `&filter_nml` | `inf_flavor` (0 none, 2 spatially varying Gaussian, 5 inverse-gamma) |
| Inflation bounds / damping / sd | `&filter_nml` | `inf_lower_bound`, `inf_damping`, `inf_sd_initial`, ... |
| Model error (forcing) | `&model_nml` | `forcing` |
| Observing network | `&perfect_model_obs_nml` | `obs_seq_in_file_name` |

Suggested experiments (mirroring Sections 3 and 5):

1. `cutoff = 0.4`, run `filter`, look at the error.
2. `ens_size = 80` with the same cutoff; then `ens_size = 10` and vary
   `cutoff` — the Section 3 story, now in Fortran.
3. Turn on adaptive inflation: `inf_flavor = 5` (first column = prior
   inflation), rerun, compare.
4. Imperfect model: `perfect_model_obs` with `forcing = 8.0`, then
   `filter` with `forcing = 10.0`.

## Choosing the filter algorithm: QCEFF tables

Section 4's distribution choices are selected in DART with a QCEFF table
(`&algorithm_info_nml :: qceff_table_filename`):

* `eakf_qceff_table.csv` — normal distributions everywhere (EAKF),
* `bnrhf_qceff_table.csv` — bounded normal rank histogram filter,
* `enkf_qceff_table.csv` — perturbed-observation EnKF.

Run the same OSSE with the EAKF and BNRHF tables and compare errors —
on Lorenz 96 (unbounded, weakly non-Gaussian) they are close; on bounded
quantities the BNRHF wins.

## Observation-space diagnostics

`obs_diag` summarizes `obs_seq.final` into `obs_diag_output.nc`:
rank histograms, RMSE/bias/spread evolution by observation type and
region — the production version of the tutorial's diagnostic panels.

## Quality control

`&quality_control_nml :: outlier_threshold` (e.g. 3.0) rejects any
observation whose innovation exceeds that many expected-separation
standard deviations — protection against gross errors. Compare the number
of observations *used* vs *available* in the obs-space diagnostics after
setting it.

## Where next

* `DART/models/lorenz_63`, `lorenz_96_tracer` (a bounded tracer —
  QCEFF territory), and dozens of real geophysical models.
* The DART documentation: https://docs.dart.ucar.edu
* Questions: dart@ucar.edu

*This completes the pyDARTLAB tutorial.*